# EE 451: Communications Systems
## Lesson 15 - Noise Fundamentals & SNR

### Learning Objectives
By the end of this lesson, you will be able to:
- Explain thermal noise generation and statistical properties
- Define and calculate noise power spectral density
- Analyze Additive White Gaussian Noise (AWGN) channel model
- Calculate signal-to-noise ratio (SNR) in communications systems
- Apply noise figure and noise temperature concepts to receiver design
- Compute cascaded noise figure for multi-stage systems

### Textbook Reference
Haykin & Moher, Chapter 9

## Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.fft import fft, fftfreq
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['lines.linewidth'] = 2

# Physical constants
k_B = 1.38e-23   # Boltzmann's constant (J/K)
T0 = 290          # Reference temperature (K)

print("Setup complete!")
print(f"Boltzmann's constant: k = {k_B:.2e} J/K")
print(f"Reference temperature: T0 = {T0} K")

## Part 1: Thermal Noise Fundamentals

**Thermal noise** (Johnson-Nyquist noise) arises from random motion of electrons in conductors.

**Noise power:** $P_n = kTB$
- $k = 1.38 \times 10^{-23}$ J/K (Boltzmann's constant)
- $T$ = temperature (Kelvin)
- $B$ = bandwidth (Hz)

**Noise voltage (RMS):** $V_n = \sqrt{4kTRB}$
- $R$ = resistance (Ω)

At room temperature ($T = 290$ K): $P_n = 4 \times 10^{-21} B$ watts

In [ ]:
# === Thermal Noise: Power vs Bandwidth and Noise Voltage vs Resistance ===

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: Noise power vs bandwidth at different temperatures ---
bandwidths = np.logspace(3, 9, 200)  # 1 kHz to 1 GHz
temperatures = [77, 290, 500, 1000]  # K (liquid N2, room, warm, hot)
temp_labels = ['77 K (LN\u2082)', '290 K (Room)', '500 K', '1000 K']

for T, label in zip(temperatures, temp_labels):
    P_n = k_B * T * bandwidths
    P_n_dBm = 10 * np.log10(P_n / 1e-3)
    axes[0].semilogx(bandwidths, P_n_dBm, linewidth=2, label=label)

axes[0].set_xlabel('Bandwidth (Hz)')
axes[0].set_ylabel('Noise Power (dBm)')
axes[0].set_title('Thermal Noise Power vs Bandwidth')
axes[0].legend()
axes[0].set_xlim([1e3, 1e9])

# --- Plot 2: Noise voltage vs resistance ---
resistances = np.logspace(0, 6, 200)  # 1 \u03a9 to 1 M\u03a9
bw_examples = [1e3, 1e6, 1e9]
bw_labels = ['B = 1 kHz', 'B = 1 MHz', 'B = 1 GHz']

for B, label in zip(bw_examples, bw_labels):
    V_n = np.sqrt(4 * k_B * T0 * resistances * B)
    axes[1].loglog(resistances, V_n * 1e6, linewidth=2, label=label)

axes[1].set_xlabel('Resistance (\u03a9)')
axes[1].set_ylabel('Noise Voltage (\u03bcV RMS)')
axes[1].set_title('Johnson-Nyquist Noise Voltage (T = 290 K)')
axes[1].legend()

plt.tight_layout()
plt.show()

# --- Worked Example ---
B_ex = 10e6  # 10 MHz
P_n_ex = k_B * T0 * B_ex
R_ex = 50  # ohms
V_n_ex = np.sqrt(4 * k_B * T0 * R_ex * B_ex)

print("=== Worked Example: Thermal Noise ===\n")
print(f"Bandwidth:   B = {B_ex/1e6:.0f} MHz")
print(f"Temperature: T = {T0} K (room temperature)")
print(f"Resistance:  R = {R_ex} \u03a9")
print(f"\nNoise power: P_n = kTB = {P_n_ex:.2e} W")
print(f"             P_n = {10*np.log10(P_n_ex/1e-3):.1f} dBm")
print(f"\nNoise voltage: V_n = \u221a(4kTRB) = {V_n_ex*1e6:.2f} \u03bcV RMS")

## Part 2: Additive White Gaussian Noise (AWGN)

The AWGN channel model is the most fundamental noise model:

$$r(t) = s(t) + n(t)$$

where $n(t)$ is **Additive**, **White**, **Gaussian** noise:

| Property | Meaning |
|----------|--------|
| **Additive** | Noise adds to signal: $r = s + n$ |
| **White** | Flat power spectral density: $S_n(f) = N_0/2$ |
| **Gaussian** | Amplitude follows Gaussian PDF: $p(n) = \frac{1}{\sqrt{2\pi\sigma^2}} e^{-n^2/2\sigma^2}$ |

**Why Gaussian?** Central Limit Theorem — sum of many independent noise sources → Gaussian.

In [ ]:
# === AWGN Visualization: Time Domain, Histogram, PSD, Autocorrelation ===
np.random.seed(42)

fs = 100e3     # 100 kHz sampling rate
duration = 0.1  # 100 ms
N = int(fs * duration)
t = np.arange(N) / fs

sigma = 1.0
noise = sigma * np.random.randn(N)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- Time domain ---
axes[0, 0].plot(t[:500] * 1e3, noise[:500], linewidth=0.5, color='steelblue')
axes[0, 0].set_xlabel('Time (ms)')
axes[0, 0].set_ylabel('Amplitude')
axes[0, 0].set_title('AWGN Time Domain')
axes[0, 0].axhline(y=0, color='k', linestyle='-', alpha=0.3)

# --- Histogram vs Gaussian PDF ---
axes[0, 1].hist(noise, bins=60, density=True, alpha=0.7, color='steelblue', label='Histogram')
x_pdf = np.linspace(-4*sigma, 4*sigma, 300)
pdf_theory = stats.norm.pdf(x_pdf, 0, sigma)
axes[0, 1].plot(x_pdf, pdf_theory, 'r-', linewidth=2.5, label='Gaussian PDF')
axes[0, 1].set_xlabel('Amplitude')
axes[0, 1].set_ylabel('Probability Density')
axes[0, 1].set_title('Amplitude Distribution')
axes[0, 1].legend()

# --- Power Spectral Density (Welch) ---
freqs, psd = signal.welch(noise, fs, nperseg=1024)
axes[1, 0].plot(freqs / 1e3, psd, linewidth=1, color='steelblue')
axes[1, 0].axhline(y=np.mean(psd), color='r', linestyle='--', linewidth=2,
                    label=f'Mean PSD = {np.mean(psd):.4f} V\u00b2/Hz')
axes[1, 0].set_xlabel('Frequency (kHz)')
axes[1, 0].set_ylabel('PSD (V\u00b2/Hz)')
axes[1, 0].set_title('Power Spectral Density (White = Flat)')
axes[1, 0].legend()

# --- Autocorrelation ---
max_lag = 100
autocorr = np.correlate(noise[:2000], noise[:2000], mode='full')
autocorr = autocorr / autocorr[len(autocorr)//2]  # normalize
mid = len(autocorr) // 2
lag_axis = np.arange(-max_lag, max_lag + 1)
axes[1, 1].plot(lag_axis, autocorr[mid - max_lag:mid + max_lag + 1],
                linewidth=1, color='steelblue')
axes[1, 1].set_xlabel('Lag (samples)')
axes[1, 1].set_ylabel('Normalized Autocorrelation')
axes[1, 1].set_title('Autocorrelation (White = Impulse at lag 0)')

plt.tight_layout()
plt.show()

print("=== AWGN Statistical Properties ===")
print(f"Mean:     {np.mean(noise):.4f}  (expected: 0)")
print(f"Std dev:  {np.std(noise):.4f}  (expected: {sigma})")
print(f"Variance: {np.var(noise):.4f}  (expected: {sigma**2})")
print(f"\nPSD is approximately flat \u2192 White noise confirmed")
print(f"Autocorrelation is impulse-like \u2192 Samples are uncorrelated")

## Part 3: Signal-to-Noise Ratio (SNR)

**SNR** quantifies signal quality relative to noise:

$$\text{SNR} = \frac{P_{\text{signal}}}{P_{\text{noise}}}$$

$$\text{SNR (dB)} = 10 \log_{10}\left(\frac{P_s}{P_n}\right) = P_s\text{(dBm)} - P_n\text{(dBm)}$$

**$E_b/N_0$ (Energy per bit to noise PSD):**

$$\frac{E_b}{N_0} = \frac{P_s / R_b}{N_0} = \frac{\text{SNR} \cdot B}{R_b}$$

- More fundamental than SNR for digital communications
- Relates directly to bit error rate (BER)

In [ ]:
# === SNR Demonstration: Signal at Various SNR Levels ===
np.random.seed(42)

f_sig = 1000   # 1 kHz signal
fs_snr = 50000 # 50 kHz sampling
t_snr = np.arange(0, 0.01, 1/fs_snr)  # 10 ms

A = 1.0
signal_clean = A * np.sin(2 * np.pi * f_sig * t_snr)
P_s = A**2 / 2  # sinusoid average power

snr_dB_values = [30, 20, 10, 3]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for i, snr_dB in enumerate(snr_dB_values):
    snr_lin = 10**(snr_dB / 10)
    P_n = P_s / snr_lin
    noise_snr = np.sqrt(P_n) * np.random.randn(len(t_snr))
    received = signal_clean + noise_snr

    axes[i].plot(t_snr*1e3, received, linewidth=0.5, alpha=0.8, label='Signal + Noise')
    axes[i].plot(t_snr*1e3, signal_clean, 'r-', linewidth=1.5, alpha=0.6, label='Clean Signal')
    axes[i].set_xlabel('Time (ms)')
    axes[i].set_ylabel('Amplitude')
    axes[i].set_title(f'SNR = {snr_dB} dB')
    axes[i].legend(fontsize=9)
    axes[i].set_ylim([-3, 3])

plt.suptitle('Effect of SNR on Signal Quality', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# --- Worked Example: SNR and Eb/N0 ---
P_s_ex = 1e-6   # 1 uW
B_snr = 10e6     # 10 MHz bandwidth
P_n_snr = k_B * T0 * B_snr
SNR_lin = P_s_ex / P_n_snr
SNR_dB = 10 * np.log10(SNR_lin)

R_b = 1e6        # 1 Mbps
N0 = k_B * T0    # noise PSD
Eb = P_s_ex / R_b
EbN0 = Eb / N0

print("=== SNR Calculation Example ===\n")
print(f"Signal power: P_s = {P_s_ex*1e6:.1f} \u03bcW = {10*np.log10(P_s_ex/1e-3):.0f} dBm")
print(f"Bandwidth:    B   = {B_snr/1e6:.0f} MHz")
print(f"Noise power:  P_n = kTB = {P_n_snr:.2e} W = {10*np.log10(P_n_snr/1e-3):.1f} dBm")
print(f"SNR = {SNR_lin:.2e} = {SNR_dB:.1f} dB")
print(f"\n=== Eb/N0 ===\n")
print(f"Bit rate: R_b = {R_b/1e6:.0f} Mbps")
print(f"Eb = P_s/R_b = {Eb:.2e} J")
print(f"N0 = kT = {N0:.2e} W/Hz")
print(f"Eb/N0 = {EbN0:.1f} = {10*np.log10(EbN0):.1f} dB")

## Part 4: Noise Figure & Cascaded Systems (Friis Formula)

**Noise Figure** quantifies how much a device degrades SNR:

$$F = \frac{\text{SNR}_{\text{in}}}{\text{SNR}_{\text{out}}} \geq 1 \qquad \text{NF (dB)} = 10\log_{10}(F)$$

**Friis Formula** for cascaded stages:

$$F_{\text{total}} = F_1 + \frac{F_2 - 1}{G_1} + \frac{F_3 - 1}{G_1 G_2} + \cdots$$

**Key insight:** The **first stage dominates** system noise figure when $G_1$ is large.

In [ ]:
# === Noise Figure: Parametric Analysis and Friis Formula ===

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: System NF vs first-stage NF ---
G1_dB = 20
G1_lin = 10**(G1_dB / 10)
NF1_range = np.linspace(0.5, 6, 100)
F1_range = 10**(NF1_range / 10)
NF2_vals = [3, 6, 10, 15]

for NF2 in NF2_vals:
    F2 = 10**(NF2 / 10)
    F_sys = F1_range + (F2 - 1) / G1_lin
    NF_sys = 10 * np.log10(F_sys)
    axes[0].plot(NF1_range, NF_sys, linewidth=2, label=f'NF$_2$ = {NF2} dB')

axes[0].plot(NF1_range, NF1_range, 'k--', alpha=0.5, label='NF$_1$ only')
axes[0].set_xlabel('First Stage NF (dB)')
axes[0].set_ylabel('System NF (dB)')
axes[0].set_title(f'System NF vs First Stage NF (G$_1$ = {G1_dB} dB)')
axes[0].legend(fontsize=9)

# --- Plot 2: System NF vs first-stage gain ---
G1_range_dB = np.linspace(0, 40, 100)
G1_range_lin = 10**(G1_range_dB / 10)
NF1_fixed = 2.0
F1_fixed = 10**(NF1_fixed / 10)

for NF2 in NF2_vals:
    F2 = 10**(NF2 / 10)
    F_sys = F1_fixed + (F2 - 1) / G1_range_lin
    NF_sys = 10 * np.log10(F_sys)
    axes[1].plot(G1_range_dB, NF_sys, linewidth=2, label=f'NF$_2$ = {NF2} dB')

axes[1].axhline(y=NF1_fixed, color='k', linestyle='--', alpha=0.5,
                label=f'NF$_1$ = {NF1_fixed} dB')
axes[1].set_xlabel('First Stage Gain (dB)')
axes[1].set_ylabel('System NF (dB)')
axes[1].set_title(f'System NF vs First Stage Gain (NF$_1$ = {NF1_fixed} dB)')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

# --- Worked Example: 3-Stage Cascaded Receiver ---
stages = [
    ('LNA',    20,  2.0),
    ('Mixer',  -7,  7.0),
    ('IF Amp', 40,  5.0),
]

print("=== Cascaded Receiver: Friis Formula ===\n")
print(f"{'Stage':<10} {'Gain (dB)':<12} {'NF (dB)':<10} {'F (linear)':<14} {'G (linear)':<14}")
print("-" * 60)

gains = []
nfs = []
for name, g_dB, nf_dB in stages:
    g_lin = 10**(g_dB / 10)
    f_lin = 10**(nf_dB / 10)
    gains.append(g_lin)
    nfs.append(f_lin)
    print(f"{name:<10} {g_dB:<12.1f} {nf_dB:<10.1f} {f_lin:<14.3f} {g_lin:<14.2f}")

# Friis formula
F_total = nfs[0]
terms = [nfs[0]]
G_product = 1
for i in range(1, len(stages)):
    G_product *= gains[i - 1]
    term = (nfs[i] - 1) / G_product
    terms.append(term)
    F_total += term

NF_total = 10 * np.log10(F_total)
Te_total = (F_total - 1) * T0

print(f"\nFriis: F_total = F\u2081 + (F\u2082-1)/G\u2081 + (F\u2083-1)/(G\u2081\u00b7G\u2082)")
print(f"       F_total = {terms[0]:.3f} + {terms[1]:.4f} + {terms[2]:.4f} = {F_total:.4f}")
print(f"       NF_total = {NF_total:.2f} dB")
print(f"       T_e = (F-1)\u00b7T\u2080 = {Te_total:.0f} K")
print(f"\nLNA contribution: {terms[0]/F_total*100:.1f}% of total noise figure")
print("\u2192 First stage (LNA) dominates! Use low-noise LNA for best performance.")

## Part 5: Effect of Noise on Modulated Signals

Noise affects different modulation types in different ways:

| Modulation | Noise Effect | Resilience |
|-----------|-------------|------------|
| **AM** | Directly corrupts envelope | Low |
| **FM** | Affects instantaneous frequency | High (FM improvement) |
| **BPSK** | Can flip bit decisions | Depends on $E_b/N_0$ |

In [ ]:
# === Effect of Noise on AM, FM, and BPSK Signals ===
np.random.seed(42)

fc = 1000    # carrier frequency (Hz)
fm = 100     # message frequency (Hz)
fs_mod = 50000
t_mod = np.arange(0, 0.05, 1/fs_mod)

snr_demo = 10  # 10 dB

fig, axes = plt.subplots(3, 2, figsize=(14, 12))

# --- AM Signal ---
m_am = 0.5 * np.cos(2 * np.pi * fm * t_mod)
s_am = (1 + m_am) * np.cos(2 * np.pi * fc * t_mod)
P_am = np.mean(s_am**2)
noise_am = np.sqrt(P_am / 10**(snr_demo/10)) * np.random.randn(len(t_mod))

axes[0, 0].plot(t_mod*1e3, s_am, linewidth=0.5, color='C0')
axes[0, 0].set_title('AM Signal (Clean)')
axes[0, 0].set_ylabel('Amplitude')
axes[0, 1].plot(t_mod*1e3, s_am + noise_am, linewidth=0.5, color='C0')
axes[0, 1].set_title(f'AM + Noise (SNR = {snr_demo} dB)')

# --- FM Signal ---
kf = 500
phase_fm = 2*np.pi*kf * np.cumsum(np.cos(2*np.pi*fm*t_mod)) / fs_mod
s_fm = np.cos(2 * np.pi * fc * t_mod + phase_fm)
P_fm = np.mean(s_fm**2)
noise_fm = np.sqrt(P_fm / 10**(snr_demo/10)) * np.random.randn(len(t_mod))

axes[1, 0].plot(t_mod*1e3, s_fm, linewidth=0.5, color='C1')
axes[1, 0].set_title('FM Signal (Clean)')
axes[1, 0].set_ylabel('Amplitude')
axes[1, 1].plot(t_mod*1e3, s_fm + noise_fm, linewidth=0.5, color='C1')
axes[1, 1].set_title(f'FM + Noise (SNR = {snr_demo} dB)')

# --- BPSK Signal ---
R_b = 200
bits = np.array([1, 0, 1, 1, 0, 0, 1, 0, 1, 0])
symbols = 2 * bits - 1
spb = int(fs_mod / R_b)
bpsk_bb = np.repeat(symbols, spb)
t_bpsk = np.arange(len(bpsk_bb)) / fs_mod
s_bpsk = bpsk_bb * np.cos(2 * np.pi * fc * t_bpsk)
P_bpsk = np.mean(s_bpsk**2)
noise_bpsk = np.sqrt(P_bpsk / 10**(snr_demo/10)) * np.random.randn(len(t_bpsk))

axes[2, 0].plot(t_bpsk*1e3, s_bpsk, linewidth=0.5, color='C2')
axes[2, 0].set_title('BPSK Signal (Clean)')
axes[2, 0].set_xlabel('Time (ms)')
axes[2, 0].set_ylabel('Amplitude')
axes[2, 1].plot(t_bpsk*1e3, s_bpsk + noise_bpsk, linewidth=0.5, color='C2')
axes[2, 1].set_title(f'BPSK + Noise (SNR = {snr_demo} dB)')
axes[2, 1].set_xlabel('Time (ms)')

plt.suptitle('Noise Effects on Different Modulation Types',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("=== Key Observations ===\n")
print("AM:   Noise directly corrupts envelope \u2192 Poor noise performance")
print("FM:   Noise affects phase/frequency \u2192 FM improvement (wideband FM)")
print("BPSK: Noise can flip bit decisions \u2192 BER = Q(\u221a(2Eb/N0))")

## Summary

### Key Formulas

| Concept | Formula |
|---------|--------|
| Thermal noise power | $P_n = kTB$ |
| Noise voltage | $V_n = \sqrt{4kTRB}$ |
| White noise PSD | $S_n(f) = N_0/2$ |
| SNR | $\text{SNR} = P_s / P_n$ |
| SNR (dB) | $\text{SNR}_{\text{dB}} = 10\log_{10}(P_s/P_n)$ |
| Energy per bit | $E_b/N_0 = (\text{SNR} \cdot B)/R_b$ |
| Noise figure | $F = \text{SNR}_{\text{in}} / \text{SNR}_{\text{out}}$ |
| Friis formula | $F_{\text{total}} = F_1 + (F_2-1)/G_1 + (F_3-1)/(G_1 G_2) + \cdots$ |

### Key Takeaways

1. **Thermal noise is fundamental** — cannot be eliminated (exists whenever $T > 0$ K)
2. **AWGN model** is the standard channel model: additive, white, Gaussian
3. **SNR in dB** makes calculations easier: addition replaces multiplication
4. **First stage dominates** system noise figure (Friis formula) → Use low-noise LNA
5. **FM is more resilient** to noise than AM; digital depends on $E_b/N_0$

### Next Topics
- Lesson 16: SNR analysis and link budgets